# 🎙️ Ma Voix — version rapide sur GPU (Google Colab)

Ce notebook fait tourner **ma-voix** sur une carte graphique gratuite de Google (NVIDIA T4)
au lieu du processeur de ton PC : chaque phrase sort en quelques secondes au lieu de plus d'une minute.

**Avant de commencer :** vérifie le menu **Exécution → Modifier le type d'exécution → GPU T4**
(normalement déjà choisi), puis **Enregistrer**.

Ensuite, clique sur **▶** à gauche de chaque cellule, **une par une, dans l'ordre**,
et attends qu'elle soit finie (✓ vert à gauche) avant de passer à la suivante.

> ⚠️ **Vie privée** : ici, ta voix et tes textes sont envoyés sur un ordinateur de Google
> (pas sur ton PC). Tout est effacé quand la session Colab se termine.
> Utilise **uniquement ta propre voix** : les règles de Colab interdisent les *deepfakes*.

**Deux façons de l'utiliser :**

- **Mode A — dans le notebook** *(recommandé, autorisé avec Colab gratuit)* : tu envoies ta voix
  et tu tapes tes textes directement dans ce notebook (cellules 4 et 5).
- **Mode B — avec ta page habituelle** : un tunnel relie ce notebook à la page ma-voix de ton PC.
  ⚠️ Les règles de Colab **gratuit** interdisent de « contourner l'interface du notebook pour
  passer surtout par une interface web » : Google peut couper la session sans prévenir.
  Réserve ce mode à un compte Colab avec des unités de calcul (payant).

In [ ]:
# 1. Verifier que la carte graphique (GPU) est bien la
import subprocess

resultat = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if resultat.returncode != 0:
    raise SystemExit("❌ Pas de carte graphique. Menu Exécution > Modifier le type d'exécution > GPU T4, "
                     "puis relance cette cellule.")
print("✅ Carte graphique trouvée :", resultat.stdout.strip())

In [ ]:
# 2. Telecharger ma-voix et installer ce qu'il faut (~3 a 5 minutes, une fois par session)
import os
import subprocess

DEPOT = "https://github.com/yassinechkir38-del/ma-voix"
DOSSIER = "/content/ma-voix"
PYTHON = "/content/venv/bin/python"  # un Python 3.11 a part, comme sur ton PC


def etape(message, commande):
    print("⏳", message)
    resultat = subprocess.run(commande, shell=True, capture_output=True, text=True)
    if resultat.returncode != 0:
        print(resultat.stdout[-3000:], resultat.stderr[-3000:])
        raise SystemExit("❌ Cette étape a échoué (détails ci-dessus).")


if os.path.isdir(DOSSIER):
    etape("Mise à jour du code de ma-voix", f"git -C {DOSSIER} pull --ff-only")
else:
    etape("Téléchargement du code de ma-voix", f"git clone --depth 1 {DEPOT} {DOSSIER}")
etape("Installation de l'outil uv", "pip install -q uv")
if not os.path.isdir("/content/venv"):
    etape("Création d'un Python 3.11 séparé", "uv venv --python 3.11 /content/venv")
# Sur Linux, torch 2.6.0 (la version de ton PC) est livre avec CUDA : il sait utiliser le GPU
etape("Installation de PyTorch pour GPU (le plus long)",
      f"uv pip install -q --python {PYTHON} torch==2.6.0 torchaudio==2.6.0")
etape("Installation de Chatterbox, Flask, etc.",
      f"uv pip install -q --python {PYTHON} -r {DOSSIER}/backend/requirements.txt")

verification = subprocess.run([PYTHON, "-c", "import torch; print(torch.cuda.is_available())"],
                              capture_output=True, text=True)
if verification.stdout.strip() != "True":
    print(verification.stdout, verification.stderr[-3000:])
    raise SystemExit("❌ PyTorch ne voit pas le GPU.")
print("✅ Installation terminée, PyTorch voit le GPU.")

## 3. Démarrer le serveur ma-voix

Choisis le mode dans la liste à droite (**A** si tu hésites), puis clique sur ▶.
La première fois, le modèle d'IA (~3 Go) se télécharge : compte quelques minutes.

Tu peux relancer cette cellule à tout moment (par exemple pour changer de mode) :
elle arrête d'abord l'ancien serveur.

In [ ]:
# 3. Demarrer ma-voix sur le GPU
mode = "A - dans le notebook"  # @param ["A - dans le notebook", "B - avec ma page (tunnel)"]

import hashlib
import os
import re
import secrets
import subprocess
import time
import urllib.request

import requests

DOSSIER = "/content/ma-voix"
PYTHON = "/content/venv/bin/python"
LOCAL = "http://127.0.0.1:5000"  # le serveur n'ecoute que sur cette machine (regle MV-1)
JOURNAL = "/content/ma_voix_serveur.log"
JOURNAL_TUNNEL = "/content/tunnel.log"
# cloudflared : version epinglee, verifiee par son empreinte SHA-256 (publiee par Cloudflare)
URL_CLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64"
SHA256_CLOUDFLARED = "77e26d8d900e0b8469f416239d14b5f296525fdf79fee6f511ef55609e3fbac2"
CLOUDFLARED = "/content/cloudflared"


def journal(lignes=30):
    """Affiche la fin du journal du serveur (utile en cas de probleme)."""
    if os.path.exists(JOURNAL):
        print("".join(open(JOURNAL, encoding="utf-8", errors="replace").readlines()[-lignes:]))


def arreter(message):
    journal()
    raise SystemExit("❌ " + message)


# Si on relance la cellule : on arrete d'abord l'ancien serveur et l'ancien tunnel
subprocess.run(["pkill", "-f", f"{DOSSIER}/backend/app.py"])
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
time.sleep(2)

# Jeton secret, nouveau a chaque demarrage : sans lui, le serveur refuse toute requete
JETON = secrets.token_urlsafe(32)
ENTETES = {"X-Ma-Voix-Jeton": JETON}
env = dict(os.environ, MA_VOIX_JETON=JETON)
env.pop("PYTHONPATH", None)  # le Python 3.11 a part ne doit pas voir les modules de Colab

TUNNEL = mode.startswith("B")
if TUNNEL:
    if not os.path.exists(CLOUDFLARED):
        print("⏳ Téléchargement de cloudflared (le tunnel)")
        urllib.request.urlretrieve(URL_CLOUDFLARED, CLOUDFLARED)
    with open(CLOUDFLARED, "rb") as f:
        if hashlib.sha256(f.read()).hexdigest() != SHA256_CLOUDFLARED:
            os.remove(CLOUDFLARED)
            raise SystemExit("❌ Le fichier cloudflared n'est pas celui attendu : arrêt par sécurité.")
    os.chmod(CLOUDFLARED, 0o755)
    print("⏳ Ouverture du tunnel")
    subprocess.Popen([CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", LOCAL],
                     stdout=open(JOURNAL_TUNNEL, "w"), stderr=subprocess.STDOUT)
    ADRESSE = None
    for _ in range(60):
        time.sleep(1)
        trouve = re.search(r"https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com", open(JOURNAL_TUNNEL).read())
        if trouve:
            ADRESSE = trouve.group(0)
            break
    if ADRESSE is None:
        print(open(JOURNAL_TUNNEL).read()[-3000:])
        raise SystemExit("❌ Le tunnel n'a pas démarré (détails ci-dessus). Relance cette cellule.")
    # Le serveur n'acceptera que l'adresse du tunnel en plus des adresses locales (regle MV-7)
    env["MA_VOIX_HOTES"] = ADRESSE.removeprefix("https://")

print("⏳ Démarrage du serveur")
serveur = subprocess.Popen([PYTHON, "-u", f"{DOSSIER}/backend/app.py"], cwd=f"{DOSSIER}/backend", env=env,
                           stdout=open(JOURNAL, "w"), stderr=subprocess.STDOUT)


def statut():
    return requests.get(f"{LOCAL}/api/statut", headers=ENTETES, timeout=10).json()


for _ in range(180):
    if serveur.poll() is not None:
        arreter("Le serveur s'est arrêté (journal ci-dessus).")
    try:
        statut()
        break
    except requests.RequestException:
        time.sleep(1)
else:
    arreter("Le serveur ne répond pas.")

if TUNNEL:
    # Verification a travers Internet : sans jeton -> refuse (403), avec le jeton -> accepte (200)
    sans_jeton = avec_jeton = None
    for _ in range(30):
        try:
            sans_jeton = requests.get(f"{ADRESSE}/api/statut", timeout=10).status_code
            avec_jeton = requests.get(f"{ADRESSE}/api/statut", headers=ENTETES, timeout=10).status_code
            if avec_jeton == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    if (sans_jeton, avec_jeton) != (403, 200):
        arreter(f"Le tunnel ne fonctionne pas comme prévu (sans jeton : {sans_jeton}, avec : {avec_jeton}).")
    print("✅ Tunnel prêt et protégé par le jeton.\n")
    print("Sur ton PC, mets ces 2 lignes dans le fichier ma-voix/frontend/.env.local (voir le README) :\n")
    print(f"VITE_API_URL={ADRESSE}")
    print(f"MA_VOIX_JETON={JETON}")
    print("\nNe partage jamais ce jeton. Il change à chaque lancement de cette cellule.\n")

print("⏳ Chargement du modèle sur le GPU (la 1re fois : téléchargement de ~3 Go)…")
debut = time.time()
while True:
    if serveur.poll() is not None:
        arreter("Le serveur s'est arrêté (journal ci-dessus).")
    try:
        etat = statut()
    except requests.RequestException:
        time.sleep(5)
        continue
    if etat["erreur"]:
        arreter(etat["erreur"])
    if etat["modele_pret"]:
        break
    if time.time() - debut > 1800:
        arreter("Le modèle met trop de temps à charger.")
    time.sleep(5)
if not etat["gpu"]:
    print("⚠️ Le modèle tourne sur le processeur, pas sur le GPU : vérifie le type d'exécution (cellule 1).")
print(f"✅ Modèle prêt en {time.time() - debut:.0f} s.",
      "Passe à la cellule 4." if not TUNNEL else "Lance maintenant la page sur ton PC (README, mode B).")

## Mode A — 4. Envoyer ta voix

1. Coche la case **c_est_ma_voix** à droite (obligatoire).
2. Clique sur ▶, puis sur **Parcourir…** / **Choisir des fichiers** et choisis un enregistrement
   de ta voix de **10 à 20 secondes**, au calme.

Tous les formats marchent (wav, mp3, m4a de l'Enregistreur vocal de Windows…).
Si tu as déjà utilisé ma-voix sur ton PC, ta voix nettoyée est dans
`Documents/dev/ma-voix/backend/voix/ma_voix.wav`.

Le serveur nettoie l'enregistrement (silences, souffle, volume), comme sur ton PC.

In [ ]:
# 4. Envoyer ta voix (mode A)
c_est_ma_voix = False  # @param {type:"boolean"}

import os
import subprocess

import requests
from google.colab import files
from IPython.display import Audio, display

if not c_est_ma_voix:
    raise SystemExit("Coche d'abord la case « c_est_ma_voix » : clone uniquement ta propre voix "
                     "(ou celle d'une personne qui a donné son accord).")

envoi = files.upload()
if len(envoi) != 1:
    raise SystemExit("Choisis un seul fichier audio.")
nom = next(iter(envoi))
# Conversion en WAV mono (le serveur n'accepte que le WAV) ; 30 secondes au maximum
conversion = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", nom, "-t", "30", "-ac", "1",
                             "-ar", "24000", "/content/voix.wav"], capture_output=True, text=True)
os.remove(nom)
if conversion.returncode != 0:
    raise SystemExit("❌ Ce fichier audio ne peut pas être lu.")

with open("/content/voix.wav", "rb") as f:
    reponse = requests.post(f"{LOCAL}/api/voix", headers={**ENTETES, "X-Ma-Voix": "1"},
                            files={"audio": ("ma_voix.wav", f, "audio/wav")}, timeout=120)
os.remove("/content/voix.wav")
resultat = reponse.json()
for conseil in resultat.get("conseils", []):
    print("💡", conseil)
if not reponse.ok:
    raise SystemExit("❌ " + resultat["erreur"])
print("✅", resultat["message"], f"({resultat['duree']} s de parole gardées). Écoute-la :")
display(Audio(requests.get(f"{LOCAL}/api/voix", headers=ENTETES, timeout=30).content))

## Mode A — 5. Générer un message

Écris ton texte dans le champ **texte** à droite, choisis la langue, puis clique sur ▶.
Tu peux relancer cette cellule autant de fois que tu veux, avec d'autres textes.

- **rapide** : sur GPU, utilise déjà la meilleure qualité de son.
- **precision** : 3 essais par phrase, garde le plus ressemblant à ta voix (~3× plus long).
- Écris la ponctuation : les points et les virgules donnent les pauses.

In [ ]:
# 5. Generer un message (mode A)
texte = "Bonjour ! Ceci est un message vocal créé avec ma propre voix."  # @param {type:"string"}
langue = "fr"  # @param ["fr", "ar", "en", "es", "de", "it", "pt", "nl", "tr", "ru", "zh", "ja", "ko", "hi", "pl", "sv", "da", "no", "fi", "el", "he", "ms", "sw"]
qualite = "rapide"  # @param ["rapide", "precision"]
expressivite = 0.5  # @param {type:"slider", min:0.25, max:1.5, step:0.05}
telecharger_sur_mon_pc = True  # @param {type:"boolean"}

import os
import time

import requests
from google.colab import files
from IPython.display import Audio, display

reponse = requests.post(f"{LOCAL}/api/generer", headers={**ENTETES, "X-Ma-Voix": "1"}, timeout=30,
                        json={"texte": texte, "langue": langue, "qualite": qualite, "expressivite": expressivite})
tache = reponse.json()
if not reponse.ok:
    raise SystemExit("❌ " + tache["erreur"])

debut = time.time()
while True:
    time.sleep(1)
    etat = requests.get(f"{LOCAL}/api/taches/{tache['id']}", headers=ENTETES, timeout=30).json()
    print(f"\r⏳ {etat['prets']} phrase(s) sur {etat['total']} — {time.time() - debut:.0f} s", end="")
    if etat["fini"]:
        break
print()
if etat["erreur"]:
    journal()
    raise SystemExit("❌ " + etat["erreur"])

os.makedirs("/content/messages", exist_ok=True)
chemin = f"/content/messages/{etat['fichier']}"
with open(chemin, "wb") as f:
    f.write(requests.get(f"{LOCAL}/api/sorties/{etat['fichier']}", headers=ENTETES, timeout=60).content)
print(f"✅ Fini en {time.time() - debut:.0f} s.")
display(Audio(chemin))
if telecharger_sur_mon_pc:
    files.download(chemin)

## Mode B — avec ta page habituelle

La cellule 3 (en mode B) a affiché deux lignes `VITE_API_URL=…` et `MA_VOIX_JETON=…`.
Suis la section **« Version rapide sur GPU (Colab) »** du README de ma-voix pour les mettre
dans `frontend/.env.local` sur ton PC, puis lancer la page avec `npm run dev`.

Garde cet onglet Colab ouvert pendant que tu utilises la page : si la session s'arrête,
la page affiche « Colab injoignable ».

## En cas de problème : le journal du serveur

In [ ]:
# Affiche les dernieres lignes du journal du serveur
journal(40)

## Arrêter

Arrête le serveur et le tunnel. Pour **tout effacer** (ta voix, tes messages) :
menu **Exécution → Déconnecter et supprimer l'environnement d'exécution**.

In [ ]:
# Arreter le serveur et le tunnel
import subprocess

subprocess.run(["pkill", "-f", "/content/ma-voix/backend/app.py"])
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
print("Arrêté.")